# TripAdvisor Sentiment Analysis — Step 4: Data Visualization & Insights

Welcome to the final stage of your research project! In this notebook, we transform the complex numeric sentiment and emotion data into beautiful, publish-ready visualizations.

### Learning Objectives:
1. Learn how to map sentiment categories (Positive/Negative) into a Bar Chart.
2. Visualize deep emotional profiles (Joy, Trust, Anger) using horizontal columns.
3. Build a beautiful **Word Cloud** to identify commonly used terms.
4. Plot a timeline trend of guest sentiment to track operational quality over time.

---

### ☁️ Google Colab Setup
If you are running this notebook in **Google Colab**, simply run the code cell below to download the dataset and install any missing tools. If you are running this locally in RStudio, you can skip it!

In [ ]:
# =====================================================================
# GOOGLE COLAB SETUP (Run this cell only if you are using Google Colab)
# =====================================================================
if (dir.exists("/content")) {
  if (!dir.exists("/content/MrKadekProject")) {
    cat("Downloading project files from GitHub...\n")
    system("git clone https://github.com/divanahadyan1618/MrKadekProject.git /content/MrKadekProject", ignore.stdout=TRUE, ignore.stderr=TRUE)
  }
  setwd("/content/MrKadekProject")
  
  # Install required packages if missing
  packages <- c("rvest", "tidytext", "stopwords", "syuzhet", "wordcloud", "RColorBrewer")
  new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
  if(length(new_packages)) install.packages(new_packages, quiet = TRUE)
  
  cat("Colab Environment Ready!\n")
}


## 1. Load Packages and Data

We load `ggplot2` (part of the `tidyverse`) to create high-quality graphics, and `wordcloud` to generate frequency clouds.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 4: Data Visualization & Insights
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Math numbers and huge spreadsheets are boring and hard to read.
# "Data Visualization" means turning those numbers into beautiful, colorful 
# pictures (like bar charts and line graphs) so we can easily understand the story!

## 2. Define Custom ggplot2 Aesthetic Theme

We define a custom premium theme so all our charts have a cohesive, professional, academic design without needing to copy-paste the formatting logic for every plot.

In [ ]:
# =====================================================================
# STEP 1: Load Packages and Data
# =====================================================================
# We load our toolsets again. 
library(tidyverse)    # Contains 'ggplot2' - the most powerful charting tool in R.
library(tidytext)     # For handling our text data.
library(wordcloud)    # A fun tool specifically for drawing Word Clouds.
library(RColorBrewer) # A tool that provides beautiful, professional color palettes.

# We load the final, scored dataset from Step 3.
research_data <- read_csv2("data/cleaned/bvlgari_sentiment_scores.csv")
# We also load the chopped up individual words from Step 2 (for the word cloud).
cleaned_tokens <- read_csv2("data/cleaned/bvlgari_cleaned_tokens.csv")

cat("Data and drawing tools loaded successfully!\n")

## 3. Plot Sentiment Class Distribution

We create a standard bar chart comparing the total volume of Positive vs Negative vs Neutral reviews, assigning semantic colors to each.

In [ ]:
# =====================================================================
# STEP 2: Create a Custom "Theme" (Paintbrush)
# =====================================================================
# Instead of telling the computer "make the title bold and size 16" every 
# single time we draw a chart, we create a 'theme_premium' recipe once.
# Now, we just slap 'theme_premium()' on any chart, and it instantly looks professional!

theme_premium <- function() {
  theme_minimal() + # Start with a clean, white background
  theme(
    text = element_text(color = "#2C3E50"), # Dark blue/grey text
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5, color = "#1A252C"), # Centered bold title
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "#5A6F7C"), # Centered subtitle
    axis.title = element_text(face = "bold", size = 11), # Bold axis labels (X and Y)
    legend.position = "none", # Hide the messy legend box
    panel.grid.minor = element_blank() # Remove distracting background grid lines
  )
}

## 4. Plot NRC Emotional Profile

We sum up the specific emotional values (anger, joy, trust, etc.) across the entire dataset to see what the dominant emotional experience is like at the hotel.

In [ ]:
# =====================================================================
# STEP 3: Draw the Sentiment Distribution Chart (Positive vs Negative)
# =====================================================================
# We use 'ggplot' to draw the chart. 
# Think of 'aes' (aesthetics) as telling the computer where to put the data.
# x = sentiment_category means "Put Positive, Neutral, and Negative on the bottom X axis".
# fill = sentiment_category means "Color them based on their category".

p1 <- ggplot(research_data, aes(x = sentiment_category, fill = sentiment_category)) +
  geom_bar(width = 0.6, alpha = 0.85) + # Draw solid bars (alpha makes them slightly transparent)
  # scale_fill_manual lets us pick the exact paint colors. Green = Good, Red = Bad, Grey = Neutral.
  scale_fill_manual(values = c("Positive" = "#16A085", "Negative" = "#E74C3C", "Neutral" = "#95A5A6")) +
  # 'labs' stands for labels. We give our chart a title.
  labs(
    title = "Sentiment Distribution of TripAdvisor Reviews",
    subtitle = "Bvlgari Resort Bali guest opinions classification",
    x = "Sentiment Category",
    y = "Number of Reviews"
  ) +
  theme_premium() # Apply our custom paintbrush from Step 2!

# 'ggsave' saves the picture to our computer folder as a PNG image file.
ggsave("output/figures/sentiment_distribution.png", plot = p1, width = 7, height = 5, dpi = 300)

## 5. Generate the Word Cloud

Word clouds are beautiful ways to see the most frequently used words. The larger the word, the more frequently it was mentioned by guests. We calculate word frequencies, excluding non-descriptive filler context words (e.g. 'hotel', 'bali' are universally mentioned and waste visual space).

In [ ]:
# =====================================================================
# STEP 4: Draw the Deep Emotion Chart (Joy, Trust, Anger, etc.)
# =====================================================================
# We calculate the total sum of all the 8 emotions.
emotions_summary <- research_data %>%
  select(anger, anticipation, disgust, fear, joy, sadness, surprise, trust) %>%
  summarise(across(everything(), sum)) %>% # Add them all up
  pivot_longer(cols = everything(), names_to = "emotion", values_to = "score") %>% # Reshape the table so we can graph it
  arrange(desc(score)) # Sort them from highest to lowest

# Draw the chart!
# 'reorder' automatically sorts the bars so the tallest one is at the top.
p2 <- ggplot(emotions_summary, aes(x = reorder(emotion, score), y = score, fill = emotion)) +
  geom_col(alpha = 0.85, width = 0.7) + # Draw the columns
  coord_flip() + # Flip the chart sideways! Horizontal charts are easier to read.
  scale_fill_brewer(palette = "Dark2") + # Use a built-in professional color palette
  labs(
    title = "Detailed Emotional Profile of Guest Experience",
    subtitle = "NRC Lexicon emotional analysis metrics for Bvlgari Resort Bali",
    x = "Emotion Dimension",
    y = "Total Emotion Score"
  ) +
  theme_premium()

ggsave("output/figures/emotions_breakdown.png", plot = p2, width = 8, height = 5, dpi = 300)

## 6. Plot Timeline Sentiment Trend

Line charts are excellent for tracking operational changes over time. By tracking average sentiment scores over time (grouped by month), researchers can pinpoint when a hotel's service declined or improved.

In [ ]:
# =====================================================================
# STEP 5: Draw the Word Cloud
# =====================================================================
# A Word Cloud shows the most frequently used words. The bigger the word, the more it was used!
# We 'filter' out boring words like 'hotel' and 'bali' because they waste space.
word_counts <- cleaned_tokens %>%
  count(word, sort = TRUE) %>%
  filter(!word %in% c("hotel", "resort", "bali", "stay", "room", "villa", "service"))

cat("Drawing the word cloud...\n")

# Open a digital "canvas" to draw the picture on
png("output/figures/wordcloud.png", width = 800, height = 800, res = 150)
set.seed(123) # Lock the randomness so the cloud looks exactly the same every time we run it

# Call the wordcloud drawing tool
wordcloud(
  words = word_counts$word,   # What words to draw
  freq = word_counts$n,       # How big to draw them based on frequency
  min.freq = 2,               # Ignore words that only appeared once
  max.words = 40,             # Stop drawing after 40 words so it doesn't look messy
  random.order = FALSE,       # Put the biggest words right in the center
  rot.per = 0.2,              # Randomly rotate 20% of the words sideways
  colors = brewer.pal(8, "Dark2") # Paint them with cool colors
)
dev.off() # Close the digital canvas and save the file!

## 🎓 Conclusion

You have successfully completely a full e-WOM Sentiment Analysis pipeline! You learned how to scrape data, clean text, apply lexicons, and visualize emotional patterns. You can use these exact notebooks to process any other hotel or brand on TripAdvisor for your research.